In [2]:
import pandas as pd

# load dataset
titanic = pd.read_csv("Titanic-Dataset.csv")

# inspect dataset
print(titanic.info())
print(titanic.describe())
print(titanic.isnull().sum())
print(titanic.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.48659

all the above terms are explained in previous students data cleaning.

In [3]:
# Using .loc for condition-based filtering
elderly_passengers = titanic.loc[titanic['Age'] > 60]
print(elderly_passengers)

# Using .iloc for position-based selection
subset = titanic.iloc[:5, :4]  # first 5 rows, first 4 columns

print(subset)

     PassengerId  Survived  Pclass                                       Name  \
33            34         0       2                      Wheadon, Mr. Edward H   
54            55         0       1             Ostby, Mr. Engelhart Cornelius   
96            97         0       1                  Goldschmidt, Mr. George B   
116          117         0       3                       Connors, Mr. Patrick   
170          171         0       1                  Van der hoef, Mr. Wyckoff   
252          253         0       1                  Stead, Mr. William Thomas   
275          276         1       1          Andrews, Miss. Kornelia Theodosia   
280          281         0       3                           Duane, Mr. Frank   
326          327         0       3                  Nysveen, Mr. Johan Hansen   
438          439         0       1                          Fortune, Mr. Mark   
456          457         0       1                  Millet, Mr. Francis Davis   
483          484         1  

titanic['Age'] > 60 - creates Boolean (True/False for each row).

.loc[...] keeps only rows where the condition is True.

:5 - means rows from index 0 up to 5.
:4 - means columns from index 0 up to 4.

.iloc is used to slice rows and columns by their numeric positions.

In [5]:
# Fill missing Titanic ages with median grouped by Pclass
titanic['Age'] = titanic.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

pattern = titanic.groupby('Pclass').agg({'Fare':'mean','Age':'mean','Survived':'mean'})

print(pattern)

             Fare        Age  Survived
Pclass                                
1       84.154687  38.062130  0.629630
2       20.662183  29.825163  0.472826
3       13.675550  24.824684  0.242363


titanic.groupby('Pclass') - This splits the dataset into groups based on passenger class (Pclass = 1, 2, or 3).

['Age'] - to handle missing ages separately for each passenger class.

.transform - it lets us fill missing ages with the median of their class, while keeping the dataset intact row by row.

lambda x: ... means for each group (x), do the following operation.

x represents the Age column for one particular group of passengers.

x.median() calculates the median age of that group.

x.fillna(x.median()) replaces all missing values (NaN) in that group with the median age of that group.


.agg() helps to apply one or more summary functions (like mean, sum, count) to columns in each group.

{'Fare':'mean','Age':'mean','Survived':'mean'} - to evaluate mean fare, mean age and survival rate of a particular class.

In [8]:
# Create a lookup table
ports = pd.DataFrame({'Embarked':['C','Q','S'],'Port':['Cherbourg','Queenstown','Southampton']})

# Merge with dataset
titanic = titanic.merge(ports, on='Embarked', how='left')

print(titanic[['Embarked','Port']].head())

  Embarked         Port
0        S  Southampton
1        C    Cherbourg
2        S  Southampton
3        S  Southampton
4        S  Southampton


Here we build a small DataFrame (ports) that acts like a dictionary.

Each Embarked code is paired with its corresponding port name.

NaN means the embarkation port wasn’t recorded for that passenger.

NaN values are simply unknown and can either be left as NaN or filled with the most common port.

.merge() joins the dataset with the lookup table.

on='Embarked' tells Pandas to match rows using the Embarked column in both DataFrames.

how='left' keeps all the passengers. If there’s no match, the new Port column will show NaN.